<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI_Phase20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from decimal import Decimal
import os, json, hashlib, sqlite3, subprocess, tempfile, ast, re, shutil, sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass
MYDRIVE=Path(os.environ.get('ICHI2027_MYDRIVE','/content/drive/MyDrive')).expanduser().resolve()
assert MYDRIVE.is_dir(), 'Mount your original Google Drive or define ICHI2027_MYDRIVE. Browser uploads are not used.'
PROJECT=MYDRIVE/'NeuroFHIR_ICHI2027'
OUT=PROJECT/'phase20_outputs'
PRIVATE=OUT/'_PRIVATE_DO_NOT_SHARE'
PRIVATE.mkdir(parents=True,exist_ok=True)
SHAREABLE=OUT/'ICHI2027_Phase20_SHAREABLE.json'
PRIVATE_MANIFEST=PRIVATE/'ICHI2027_Phase20_PRIVATE_C_ADAPTER_MANIFEST.json'
PRIVATE_EVENTS=PRIVATE/'ICHI2027_Phase20_PRIVATE_MRI_SOURCE_AB_GENERIC_CANDIDATES.ndjson'
PRIVATE_CONTEXT=PRIVATE/'ICHI2027_Phase20_PRIVATE_TRACE_PAIR_AB_CONTEXT_CANDIDATES.ndjson'
CODE_REVIEW=PRIVATE/'ICHI2027_Phase20_PINNED_v15_CODE_REVIEW_PRIVATE.py'
V15_COMMIT='a4b7265adabf096e84b957de068e24a5ab853425'
V15_SHA='5416d658e18d9d4f3dfb229996d685bd3dbeeb1d5d9ac910443b8527e9f3ff19'
FHIR_SCHEMA_SHA='75e5560da3cf503895a44c8ca7af17a83b4cca6c2cb5ba1883d2aec0d1cb5ac6'
FIELD_SYSTEM='https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/source-field'
FEATURES={'LHIPPOC','RHIPPOC','LENTORHIN','RENTORHIN','LMIDTEMP','RMIDTEMP','VENTRICLES','ICV'}
BIO=FEATURES-{'ICV'}
PAIR_PREFIX='Research-only TRACE Q90 longitudinal subject-pair reliability for '
PACKAGE_TEXT='Research-only descriptive TRACE subject-pair package classification'

def file_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as s:
        for block in iter(lambda:s.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def canonical_number(x):
    n=Decimal(str(x));assert n.is_finite()
    return format(n.normalize(),'f')
def js_hash(x):return hashlib.sha256(json.dumps(x,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
def utcnow():return datetime.now(timezone.utc).isoformat()
print('Phase20: original v15 method inspection + ACTUAL private MRI generic-input candidate staging; NOT a claim that frozen generic Arm C has run.')


Mounted at /content/drive
Phase20: original v15 method inspection + ACTUAL private MRI generic-input candidate staging; NOT a claim that frozen generic Arm C has run.


In [ ]:
p19_path=PROJECT/'phase19_outputs'/'ICHI2027_Phase19_SHAREABLE.json'
m19_path=PROJECT/'phase19_outputs'/'_PRIVATE_DO_NOT_SHARE'/'ICHI2027_Phase19_PRIVATE_D_AUDIT_MANIFEST.json'
p18_path=PROJECT/'phase18_outputs'/'ICHI2027_Phase18_SHAREABLE.json'
m18_path=PROJECT/'phase18_outputs'/'_PRIVATE_DO_NOT_SHARE'/'ICHI2027_Phase18_PRIVATE_AB_MANIFEST.json'
p17_path=PROJECT/'phase17_outputs'/'ICHI2027_Phase17_SHAREABLE.json'
m17_path=PROJECT/'phase17_outputs'/'_PRIVATE_DO_NOT_SHARE'/'ICHI2027_Phase17_PRIVATE_LINKAGE_MANIFEST.json'
for path in (p19_path,m19_path,p18_path,m18_path,p17_path,m17_path):
    assert path.is_file(),f'REQUIRED original same-Drive input absent: {path.name}. Restore it in Drive; do not upload private artifacts here.'
p19=json.loads(p19_path.read_text(encoding='utf-8'))
m19=json.loads(m19_path.read_text(encoding='utf-8'))
p18=json.loads(p18_path.read_text(encoding='utf-8'))
m18=json.loads(m18_path.read_text(encoding='utf-8'))
p17=json.loads(p17_path.read_text(encoding='utf-8'))
m17=json.loads(m17_path.read_text(encoding='utf-8'))
assert p19['phase']==m19['phase']=='19_frozen_neuro_D_real_AB_and_synthetic_E0_E7'
assert p19['run_utc']==m19['run_utc']
assert m19['source_phase18_public_SHA256']==file_sha(p18_path)
assert m19['neuro_D_contract_SHA256']==p19['neuro_D_algorithm_contract_sha256']
assert p19['synthetic_unit_tests']['all_passed'] is True
assert p18['phase']==m18['phase']=='18_real_private_two_arm_A_B_ablation'
assert m18['run_utc']==p18['run_utc']
assert p17['phase']=='17_real_pair_level_TRACE_to_FHIR_linkage'
assert p17['fhir_r4']['official_schema_validation']=='PASS'
assert p17['fhir_r4']['official_pinned_archive_sha256']==FHIR_SCHEMA_SHA
assert m17['private_FHIR_NDJSON_sha256']==m18['source_private_FHIR_SHA256']==m19['source_phase17_FHIR_SHA256']
FHIR_PATH=Path(os.environ.get('ICHI2027_PHASE17_PRIVATE_FHIR',m17['private_FHIR_NDJSON']))
DB=Path(os.environ.get('ICHI2027_PHASE18_PRIVATE_DB',m18['private_A_B_SQLite_file']))
PRIVATE_D_EVENTS=Path(m19['private_case_events_file'])
for path in (FHIR_PATH,DB,PRIVATE_D_EVENTS):assert path.is_file(),f'Original private file missing from Drive: {path.name}'
assert file_sha(FHIR_PATH)==m17['private_FHIR_NDJSON_sha256']
assert file_sha(DB)==m18['private_A_B_SQLite_SHA256']==m19['phase18_AB_SQLite_SHA256']
assert file_sha(PRIVATE_D_EVENTS)==m19['private_case_events_SHA256']
assert p18['direct_A_vs_B_invariants']['numeric_event_key_and_exact_Decimal_value_mismatches']==0
assert p19['actual_real_D_source_target_audit']['participants_audited']==258
print('Original Phase17 / Phase18 / Phase19 public and PRIVATE lineage: SHA-256 and run-identity checks PASSED.')


Original Phase17 / Phase18 / Phase19 public and PRIVATE lineage: SHA-256 and run-identity checks PASSED.


In [ ]:
# Retrieve the SAME frozen generic TFL commit and exact frozen v15 runner; do not rewrite its method.
PIN_DIR=Path('/content/ichi2027_phase20_pinned_generic_tfl')
if PIN_DIR.exists():
    try:
        old=subprocess.run(['git','rev-parse','HEAD'],cwd=str(PIN_DIR),text=True,capture_output=True,timeout=20)
        if old.returncode or old.stdout.strip()!=V15_COMMIT:shutil.rmtree(PIN_DIR)
    except Exception:shutil.rmtree(PIN_DIR)
if not PIN_DIR.exists():
    cmd=['git','clone','--quiet','https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git',str(PIN_DIR)]
    result=subprocess.run(cmd,text=True,capture_output=True,timeout=240)
    assert result.returncode==0, 'Public pinned git clone failed. Connect Colab to GitHub then rerun this cell. '+result.stderr[-500:]
checkout=subprocess.run(['git','checkout','--quiet',V15_COMMIT],cwd=str(PIN_DIR),text=True,capture_output=True,timeout=120)
assert checkout.returncode==0,'The exact frozen TFL commit was not retrievable; no substitute revision is permitted.'
actual_commit=subprocess.run(['git','rev-parse','HEAD'],cwd=str(PIN_DIR),text=True,capture_output=True,check=True,timeout=30).stdout.strip()
assert actual_commit==V15_COMMIT
RUNNER=PIN_DIR/'FHIRy_pyOMOP_TFL_SubmissionFreezeRunner_v15.ipynb'
assert RUNNER.is_file() and file_sha(RUNNER)==V15_SHA,'Original v15 notebook SHA-256 mismatch.'
runner=json.loads(RUNNER.read_text(encoding='utf-8'))
code_cells=[(i,''.join(c.get('source',[])) if isinstance(c.get('source'),list) else c.get('source','')) for i,c in enumerate(runner['cells']) if c['cell_type']=='code']
assert len(code_cells)==17
print('Genuine frozen generic TFL v15 source pinned to original git commit; exact runner SHA-256 verified; 17 source code cells.')


Genuine frozen generic TFL v15 source pinned to original git commit; exact runner SHA-256 verified; 17 source code cells.


In [ ]:
# STATIC inspection only: neither notebook outputs nor references to old medication audits establish a real MRI Arm C run.
method_funcs=[]; functions_source=[]; static_literal_inputs=set(); static_column_candidates=set(); parse_errors=0
for cell_index,source in code_cells:
    try:tree=ast.parse(source)
    except SyntaxError:parse_errors+=1;continue
    for node in ast.walk(tree):
        if isinstance(node,(ast.FunctionDef,ast.AsyncFunctionDef)):
            args=[a.arg for a in node.args.posonlyargs+node.args.args+node.args.kwonlyargs]
            method_funcs.append({'function':node.name,'arguments':args,'cell_index_zero_based':cell_index,
                'source_sha256':hashlib.sha256(ast.get_source_segment(source,node).encode()).hexdigest() if ast.get_source_segment(source,node) else None})
            if node.name in {'compute_metrics','split_warning_tokens','join_warning_tokens','warning_count','nonempty_warning'}:
                functions_source.append('# ORIGINAL v15 PUBLIC REPO, NOTEBOOK CELL '+str(cell_index)+'; not modified\n'+ast.get_source_segment(source,node))
        if isinstance(node,ast.Constant) and isinstance(node.value,str):
            s=node.value.strip()
            if re.search(r'\.(csv|parquet|json|zip)$',s,re.I):static_literal_inputs.add(Path(s.replace('\\','/')).name)
        if isinstance(node,ast.Subscript) and isinstance(node.slice,ast.Constant) and isinstance(node.slice.value,str):
            label=node.slice.value
            if len(label)<100 and re.fullmatch('[A-Za-z][A-Za-z_0-9]*',label):static_column_candidates.add(label)
assert parse_errors==0,'Original v15 source code cannot be parsed: STOP instead of guessing interface.'
assert 'compute_metrics' in {f['function'] for f in method_funcs}
code_review_head='# ORIGINAL PUBLIC TFL REPO: https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git\n# Frozen commit: '+V15_COMMIT+'\n# Runner original SHA-256: '+V15_SHA+'\n# Purely a static review copy. Do NOT execute it or treat old historical outcomes as MRI results.\n\n'
CODE_REVIEW.write_text(code_review_head+'\n\n'.join(functions_source)+'\n',encoding='utf-8')
ORIGINAL_METHOD={
 'repo_commit':actual_commit,'runner_sha256':file_sha(RUNNER),'runner_code_cell_count':len(code_cells),
 'original_functions':method_funcs,'literal_dependency_basenames':sorted(static_literal_inputs),
 'inferred_column_name_literals_NOT_VALIDATED_INTERFACE':sorted(static_column_candidates),
 'compute_metrics_static_source_present':True,
 'original_runner_prior_phase06_replay_did_NOT_apply_to_real_MRI':True,
 'review_code_path_PRIVATE':str(CODE_REVIEW),'review_code_sha256':file_sha(CODE_REVIEW)}
print('Original methods found:',[f['function'] for f in method_funcs])
print('Historical file dependencies:',len(static_literal_inputs),'; original generic input requirements remain unapproved until method-level semantic inspection.')


Original methods found: ['nonempty_warning', 'compute_metrics', 'nonempty_warning', 'compute_metrics', 'split_warning_tokens', 'join_warning_tokens', 'warning_count', 'sha256_file']
Historical file dependencies: 20 ; original generic input requirements remain unapproved until method-level semantic inspection.


In [ ]:
# Source-to-adapter contract is frozen before emitting any patient-level candidate events.
# IMPORTANT: this is a PRIVATE MRI SOURCE/TARGET NORMALIZED CANDIDATE, not already the original v15 schema.
CANDIDATE_CONTRACT={
 'version':'ICHI2027_PHASE20_REAL_MRI_GENERIC_INPUT_CANDIDATE_V1',
 'source':'SHA_verified_Phase17_private_FHIR_R4_bundles',
 'targets':'SHA_verified_Phase18_private_A_and_deliberate_B_staging',
 'analysis_unit':'subject_nested_measurement; no independent-row inference',
 'per_event_fields':['FHIR_observation_fullUrl','private_person_reference','source_feature','BL_or_M12',
   'source_date','source_decimal_value','target_decimal_value','source_ImagingStudy_ref',
   'target_image_ref_A_or_missing_B','source_unit_missing','target_unit_missing',
   'source_identifier_provenance_only_unverified_DICOM','arm'],
 'paired_TRACE_states':'source FHIR pair-level assessments, not scan-level original source metadata',
 'missing_data_semantics':'UNKNOWN_SOURCE_NOT_TRANSFORMATION_LOSS',
 'required_unresolved_for_official_OMOP':['exact_source_unit','approved_concept_id','approved_procedure_occurrence_id',
   'verified_DICOM_StudyInstanceUID','verified_DICOM_SeriesInstanceUID','Athena_vocabulary_snapshot'],
 'frozen_v15_generic_input_equivalence':'UNDETERMINED_UNTIL_ORIGINAL_FUNCTION_SEMANTIC_REVIEW',
 'never_invent':['unit','DICOM_UID','OMOP_concept','historical_20B_state']}
CANDIDATE_SHA=js_hash(CANDIDATE_CONTRACT)
con=sqlite3.connect('file:'+str(DB)+'?mode=ro',uri=True)
con.row_factory=sqlite3.Row
assert con.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
assert con.execute('PRAGMA foreign_key_check').fetchall()==[]
COUNTS={name:con.execute('SELECT COUNT(*) FROM '+name).fetchone()[0] for name in (
 'A_PERSON','B_PERSON','A_IMAGE','A_NUMERIC','B_NUMERIC','A_PAIR_STATE','A_PACKAGE_STATE','A_PACKAGE_MEMBER')}
assert COUNTS=={'A_PERSON':258,'B_PERSON':258,'A_IMAGE':516,'A_NUMERIC':4128,'B_NUMERIC':4128,
                'A_PAIR_STATE':1806,'A_PACKAGE_STATE':258,'A_PACKAGE_MEMBER':1806}
print('Frozen private MRI candidate schema hash:',CANDIDATE_SHA)
print('Existing real private A/B table integrity and audited cardinalities verified.')


Frozen private MRI candidate schema hash: f24ccfeb7c39aca5d8c005108b84badd4f40d4f374021b5dc0c33ada90f8bacc
Existing real private A/B table integrity and audited cardinalities verified.


In [ ]:
# Exact source-to-target MRI candidates on REAL input; never emit participant-level records in public JSON.
source_numeric=0;source_images=0;source_pairs=0;source_packages=0;people=0
arm_events=Counter();context_events=Counter();unknown=Counter()
part=PRIVATE_EVENTS.with_suffix('.ndjson.part')
context_part=PRIVATE_CONTEXT.with_suffix('.ndjson.part')
for p in (part,context_part):
    if p.exists():p.unlink()
try:
    with FHIR_PATH.open('r',encoding='utf-8') as inp,part.open('w',encoding='utf-8') as sink,context_part.open('w',encoding='utf-8') as context_sink:
        for raw in inp:
            if not raw.strip():continue
            bundle=json.loads(raw);entries=bundle['entry'];assert len(entries)==27
            index={x['fullUrl']:x['resource'] for x in entries};assert len(index)==27
            patient=[key for key,res in index.items() if res['resourceType']=='Patient']
            images={key:res for key,res in index.items() if res['resourceType']=='ImagingStudy'}
            observations={key:res for key,res in index.items() if res['resourceType']=='Observation'}
            assert len(patient)==1 and len(images)==2 and len(observations)==24
            person=patient[0];people+=1;source_images+=len(images)
            assert con.execute('SELECT 1 FROM A_PERSON WHERE person_key=?',(person,)).fetchone()
            assert con.execute('SELECT 1 FROM B_PERSON WHERE person_key=?',(person,)).fetchone()
            for image_ref,res in images.items():
                assert res['subject']['reference']==person
                target=con.execute('SELECT image_key,person_key,image_date,source_identifiers_JSON FROM A_IMAGE WHERE image_key=?',(image_ref,)).fetchone()
                assert target and target['person_key']==person and target['image_date']==res['started']
                assert json.loads(target['source_identifiers_JSON'])==res['identifier']
            numeric=[];pairs=[];packages=[]
            for key,obs in observations.items():
                assert obs['subject']['reference']==person
                if 'valueQuantity' in obs:numeric.append((key,obs))
                elif obs.get('code',{}).get('text','').startswith(PAIR_PREFIX):pairs.append((key,obs))
                elif obs.get('code',{}).get('text','')==PACKAGE_TEXT:packages.append((key,obs))
                else:raise AssertionError('Unrecognized original FHIR observation: STOP.')
            assert len(numeric)==16 and len(pairs)==7 and len(packages)==1
            source_numeric+=16;source_pairs+=7;source_packages+=1
            for key,obs in numeric:
                field=[x['value'] for x in obs['identifier'] if x.get('system')==FIELD_SYSTEM]
                assert len(field)==1
                tp,feature=field[0].split('_',1);assert tp in {'BL','M12'} and feature in FEATURES
                assert 'unit' not in obs['valueQuantity'] and 'code' not in obs['valueQuantity']
                refs=[r['reference'] for r in obs['derivedFrom']];assert len(refs)==1 and refs[0] in images
                source_value=canonical_number(obs['valueQuantity']['value'])
                a=con.execute('SELECT * FROM A_NUMERIC WHERE observation_key=?',(key,)).fetchone()
                b=con.execute('SELECT * FROM B_NUMERIC WHERE observation_key=?',(key,)).fetchone()
                assert a is not None and b is not None
                assert a['person_key']==b['person_key']==person
                assert a['image_key']==refs[0] and a['timepoint']==b['timepoint']==tp
                assert a['feature']==b['feature']==feature
                assert a['observation_date']==b['observation_date']==obs['effectiveDateTime']
                assert canonical_number(a['value_decimal'])==canonical_number(b['value_decimal'])==source_value
                for arm,tgt in (('A',a),('B',b)):
                    rec={'candidate_only_NOT_original_frozen_v15_schema':True,'arm':arm,
                         'FHIR_observation_fullUrl_PRIVATE':key,'private_person_reference':person,
                         'source_feature':feature,'timepoint':tp,'source_date':obs['effectiveDateTime'],
                         'source_decimal_value':source_value,'target_decimal_value':canonical_number(tgt['value_decimal']),
                         'source_image_ref_PRIVATE':refs[0],
                         'target_image_ref_PRIVATE':tgt['image_key'] if arm=='A' else None,
                         'source_numeric_unit':None,'target_numeric_unit':None,
                         'source_identifier_semantics':'UNVERIFIED_DICOM',
                         'OMOP_concept_id':None,'real_ETL_target':'PRIVATE_RESEARCH_CANDIDATE_NOT_MICDM'}
                    sink.write(json.dumps(rec,sort_keys=True,allow_nan=False)+'\n')
                    arm_events[arm]+=1
                unknown['unverified_source_numeric_unit']+=1
            for key,obs in pairs:
                feature=obs['code']['text'][len(PAIR_PREFIX):];assert feature in BIO
                target=con.execute('SELECT * FROM A_PAIR_STATE WHERE pair_key=?',(key,)).fetchone()
                assert target is not None and target['person_key']==person and target['feature']==feature
                refs=[x['reference'] for x in obs['derivedFrom']];assert refs==[target['baseline_observation_key'],target['month12_observation_key']]
                assert target['state']==obs['valueCodeableConcept']['text']
                for arm in ('A','B'):
                    context_sink.write(json.dumps({'candidate_only_NOT_original_frozen_v15_schema':True,
                        'context_kind':'REGENERATED_TRACE_SUBJECT_PAIR','arm':arm,'pair_key_PRIVATE':key,
                        'person_key_PRIVATE':person,'feature':feature,'BL_source_observation_PRIVATE':refs[0],
                        'M12_source_observation_PRIVATE':refs[1],'source_regenerated_state':target['state'],
                        'target_state':target['state'] if arm=='A' else None,
                        'source_calibration_sha256_PRIVATE':target['regenerated_calibration_CSV_SHA256'],
                        'target_calibration_sha256_PRIVATE':target['regenerated_calibration_CSV_SHA256'] if arm=='A' else None,
                        'original_historical_state':None,'not_scan_level':True},sort_keys=True)+'\n')
                    context_events[arm]+=1
            pkg_key,pkg=packages[0]
            target_pkg=con.execute('SELECT * FROM A_PACKAGE_STATE WHERE package_key=?',(pkg_key,)).fetchone()
            assert target_pkg and target_pkg['person_key']==person and target_pkg['state']==pkg['valueCodeableConcept']['text']
            assert set(x['reference'] for x in pkg['derivedFrom'])==set(r['pair_key'] for r in con.execute(
                'SELECT pair_key FROM A_PACKAGE_MEMBER WHERE package_key=?',(pkg_key,)))
            assert len(pkg['derivedFrom'])==7
            for arm in ('A','B'):
                context_sink.write(json.dumps({'candidate_only_NOT_original_frozen_v15_schema':True,
                    'context_kind':'REGENERATED_TRACE_PAIR_PACKAGE','arm':arm,'package_key_PRIVATE':pkg_key,
                    'person_key_PRIVATE':person,'source_regenerated_state':target_pkg['state'],
                    'target_state':target_pkg['state'] if arm=='A' else None,
                    'source_pair_references_PRIVATE':[r['reference'] for r in pkg['derivedFrom']],
                    'target_pair_references_PRIVATE':[r['reference'] for r in pkg['derivedFrom']] if arm=='A' else [],
                    'original_historical_state':None,'not_scan_level':True},sort_keys=True)+'\n')
                context_events[arm]+=1
        sink.flush();os.fsync(sink.fileno())
        context_sink.flush();os.fsync(context_sink.fileno())
    assert (people,source_images,source_numeric,source_pairs,source_packages)==(258,516,4128,1806,258)
    assert arm_events=={'A':4128,'B':4128}
    assert context_events=={'A':2064,'B':2064}
    os.replace(part,PRIVATE_EVENTS)
    os.replace(context_part,PRIVATE_CONTEXT)
finally:
    con.close()
print('Real MRI source/A/B normalized candidate rows privately generated:',sum(arm_events.values()),'all numeric and original FHIR pair links reconciled.')


Real MRI source/A/B normalized candidate rows privately generated: 8256 all numeric and original FHIR pair links reconciled.


In [ ]:
# A method-specific readiness matrix; source-column lexical matches DO NOT establish genuine frozen C equivalence.
BRIDGE={
 'person_identity':{'source':'FHIR Patient fullUrl','target':'candidate person_key',
                    'phase20':'EXACT_PRIVATE_PSEUDONYMOUS_MATCH','v15_semantics':'REQUIRES_ORIGINAL_RUNNER_REVIEW'},
 'numeric_source_value':{'source':'FHIR Observation.valueQuantity.value','target':'A/B value_decimal',
                         'phase20':'DECIMAL_EQUAL','v15_semantics':'REQUIRES_ORIGINAL_RUNNER_REVIEW'},
 'date_and_timepoint':{'source':'FHIR effectiveDateTime and BL/M12 local label','target':'A/B stored source date/timepoint',
                       'phase20':'MATCH','v15_semantics':'REQUIRES_ORIGINAL_RUNNER_REVIEW'},
 'source_image_link':{'source':'FHIR Observation.derivedFrom ImagingStudy','target':'Arm A retained/Arm B ablated',
                      'phase20':'EXACT_A_AND_INTENTIONAL_B_ABLATION','v15_semantics':'CANNOT_ASSUME_GENERIC_ENGINE_MODELS_IMAGE_REFERENCES'},
 'TRACE_pair_state':{'source':'Regenerated Phase16 subject-pair FHIR Assessment','target':'Arm A retains/Arm B removes',
                     'phase20':'EXACT_A_AND_INTENTIONAL_B_ABLATION','v15_semantics':'CANNOT_ASSUME_GENERIC_ENGINE_MODELS_TRACE'},
 'source_unit':{'source':'ABSENT_IN_LOCAL_EXPORT','target':'ABSENT','phase20':'SOURCE_UNKNOWN',
                'v15_semantics':'NOT_EVALUABLE_DO_NOT_COUNT_AS_ETL_LOSS'},
 'DICOM_study_and_series_UID':{'source':'UNVERIFIED_LOCAL_SOURCE_LABELS','target':'UNVERIFIED',
                               'phase20':'SOURCE_UNKNOWN','v15_semantics':'NOT_EVALUABLE'},
 'OMOP_concept_and_procedure':{'source':'NO_APPROVED_ATHENA_SNAPSHOT','target':'NO_OFFICIAL_MAPPING',
                               'phase20':'BLOCKED','v15_semantics':'CANNOT_EXECUTE_MEANINGFUL_OFFICIAL_OMOP_COMPARATOR'}}
ready_for_genuine_frozen_C=False
C_BLOCKERS=[
 'Original v15 compute_metrics contract must be inspected against its actual source-to-target event input schema; do not substitute a new warning counter.',
 'v15 frozen clinical-medication audit cannot be presumed to model imaging or regenerated TRACE pair-level evidence.',
 'MRI field-to-OMOP approved concepts, authoritative exact-export units and DICOM UID semantics are not present.',
 'Phase20 normalized MRI data are adapter CANDIDATES, not actual frozen v15 input and not an official MI-CDM load.']
print('Frozen v15 exact original source and 8,256 REAL private MRI source-to-target candidate rows ready for interface review.')
print('Genuine frozen generic Arm C executed on current real MRI:',ready_for_genuine_frozen_C)


Frozen v15 exact original source and 8,256 REAL private MRI source-to-target candidate rows ready for interface review.
Genuine frozen generic Arm C executed on current real MRI: False


In [ ]:
PRIVATE_REPORT={
 'project':p19['project'],'phase':'20_frozen_v15_generic_C_original_method_and_real_MRI_adapter_preflight',
 'run_utc':utcnow(),'source_phase19_public_SHA256':file_sha(p19_path),
 'phase19_private_manifest_SHA256':file_sha(m19_path),
 'phase17_private_fhir_sha256':file_sha(FHIR_PATH),'phase18_private_db_sha256':file_sha(DB),
 'pinned_v15':ORIGINAL_METHOD,'MRI_candidate_contract':CANDIDATE_CONTRACT,
 'MRI_candidate_contract_sha256':CANDIDATE_SHA,
 'private_real_input_candidate_file':str(PRIVATE_EVENTS),'private_real_input_candidate_SHA256':file_sha(PRIVATE_EVENTS),
 'private_context_candidate_file':str(PRIVATE_CONTEXT),'private_context_candidate_SHA256':file_sha(PRIVATE_CONTEXT),
 'exact_FHIR_private_bundles_processed':people,'source_numeric':source_numeric,'source_pair_states':source_pairs,
 'arm_candidate_event_counts':dict(arm_events),'context_candidate_event_counts':dict(context_events),
 'unknown_source_units':unknown['unverified_source_numeric_unit'],
 'v15_field_bridge':BRIDGE,'frozen_generic_v15_actual_real_MRI_execution':False,
 'C_original_code_review_required':True,'private_original_code_review_file':str(CODE_REVIEW),
 'not_official_MICDM_or_four_arm':True}
PRIVATE_MANIFEST.write_text(json.dumps(PRIVATE_REPORT,indent=2,allow_nan=False),encoding='utf-8')
SHARE={
 'project':p19['project'],'phase':PRIVATE_REPORT['phase'],'run_utc':PRIVATE_REPORT['run_utc'],
 'status':'ACTUAL_REAL_MRI_INPUT_ADAPTER_CANDIDATES_PLUS_PINNED_ORIGINAL_V15_METHOD_INSPECTION_ONLY_C_NOT_EXECUTED',
 'privacy':'Aggregate-only; private source-to-target events, subject links and original code review remain in private Google Drive.',
 'phase19_D_real_AB_and_synthetic_tests_inherited':'PASS',
 'frozen_original_generic_v15':{
   'commit_verified':True,'repository_commit':V15_COMMIT,
   'runner_sha256_verified':True,'runner_sha256':V15_SHA,'runner_code_cells':len(code_cells),
   'function_signatures':[{k:v for k,v in d.items() if k in ('function','arguments')} for d in method_funcs],
   'historical_artifact_basename_count':len(static_literal_inputs),
   'actual_original_v15_ran_on_real_current_MRI':False,
   'original_runner_phase06_run_not_current_MRI_validation':True},
 'actual_real_MRI_candidate_adapter':{
   'source_FHIR_bundles':people,'source_imaging_studies':source_images,
   'source_numeric_events':source_numeric,'regenerated_pair_feature_assessments':source_pairs,
   'regenerated_pair_packages':source_packages,
   'A_private_paired_candidate_records':arm_events['A'],
   'B_private_paired_candidate_records':arm_events['B'],
   'total_private_paired_event_records':sum(arm_events.values()),
   'A_private_pair_and_package_context_records':context_events['A'],
   'B_private_pair_and_package_context_records':context_events['B'],
   'total_private_context_records':sum(context_events.values()),
   'FHIR_to_A_B_numeric_value_or_source_key_mismatches':0,
   'FHIR_to_A_regenerated_TRACE_pair_or_package_link_errors':0,
   'original_source_unit_events_unverified':unknown['unverified_source_numeric_unit'],
   'this_is_original_v15_input':False,
   'adapter_contract_sha256':CANDIDATE_SHA},
 'v15_semantic_bridge_status':{k:v['v15_semantics'] for k,v in BRIDGE.items()},
 'gates':{
   'pinned_frozen_v15_source_and_hash_verified':True,
   'real_MRI_source_and_A_B_adapter_candidate_reconciled':True,
   'real_regenerated_TRACE_pair_state_links_reconciled':True,
   'exact_v15_generic_input_semantic_mapping_approved':False,
   'actual_genuine_frozen_v15_C_on_current_real_MRI_executed':False,
   'source_specific_unit_attestation':False,
   'DICOM_study_and_series_UID_attestation':False,
   'Athena_vocabulary_mapping_approved':False,
   'official_MICDM_load':False,
   'true_four_arm_experiment':False},
 'C_blockers':C_BLOCKERS,'four_arm_comparator_scores':None,'official_MI_CDM_load_results':None,
 'next_gate':'REVIEW_ORIGINAL_PINNED_V15_COMPUTE_METRICS_AND_INPUT_EVENT_SCHEMAS; BUILD_ONLY_SEMANTICALLY_FAITHFUL_REAL_MRI_C_ADAPTER; OBTAIN_SOURCE_AND_ATHENA_BEFORE_OFFICIAL_MICDM'}
SHAREABLE.write_text(json.dumps(SHARE,indent=2,allow_nan=False),encoding='utf-8')
print(json.dumps({'phase':SHARE['phase'],'real_adapter_rows':sum(arm_events.values()),
 'v15_source_hash_verified':True,'frozen_C_real_MRI_executed':False,'official_MICDM_executed':False},indent=2))


{
  "phase": "20_frozen_v15_generic_C_original_method_and_real_MRI_adapter_preflight",
  "real_adapter_rows": 8256,
  "v15_source_hash_verified": true,
  "frozen_C_real_MRI_executed": false,
  "official_MICDM_executed": false
}


In [ ]:
assert SHAREABLE.is_file() and PRIVATE_MANIFEST.is_file() and CODE_REVIEW.is_file() and PRIVATE_EVENTS.is_file() and PRIVATE_CONTEXT.is_file()
assert file_sha(PRIVATE_EVENTS)==json.loads(PRIVATE_MANIFEST.read_text())['private_real_input_candidate_SHA256']
assert file_sha(PRIVATE_CONTEXT)==json.loads(PRIVATE_MANIFEST.read_text())['private_context_candidate_SHA256']
recheck=json.loads(SHAREABLE.read_text(encoding='utf-8'))
assert recheck['actual_real_MRI_candidate_adapter']['total_private_paired_event_records']==8256
assert recheck['actual_real_MRI_candidate_adapter']['total_private_context_records']==4128
assert recheck['gates']['actual_genuine_frozen_v15_C_on_current_real_MRI_executed'] is False
assert recheck['four_arm_comparator_scores'] is None
print('SHAREABLE AGGREGATES ONLY:',SHAREABLE)
print('PRIVATE original frozen v15 function review:',CODE_REVIEW)
print('Do not upload private MRI candidate NDJSON or private manifest to public GitHub / this chat.')


SHAREABLE AGGREGATES ONLY: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase20_outputs/ICHI2027_Phase20_SHAREABLE.json
PRIVATE original frozen v15 function review: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase20_outputs/_PRIVATE_DO_NOT_SHARE/ICHI2027_Phase20_PINNED_v15_CODE_REVIEW_PRIVATE.py
Do not upload private MRI candidate NDJSON or private manifest to public GitHub / this chat.
